<div align="center">

<h2>ESCUELA POLITÉCNICA NACIONAL</h2>

<h3>FACULTAD DE INGENIERÍA DE SISTEMAS</h3>

<p><strong>Materia:</strong> Recuperación de la Información</p>

<p><strong>Estudiante:</strong> Estéfano Condoy</p>

</div>

<hr>

# TF, DF e IDF en 100 documentos de Gutenberg



## Objetivo

Aquí voy a seleccionar 100 libros únicos del corpus grande y calcular la frecuencia de términos con `TfidfVectorizer`. El análisis mostrará **TF** (cuántas veces aparece cada término), **DF** (en cuántos documentos aparece), **IDF** (qué tan raro es en la colección) y el peso **TF-IDF**.



## Paso 1. Preparar las bibliotecas y el tokenizador

Aquí importo las herramientas y defino la tokenización que usarán ambos vectorizadores. Si falta alguna biblioteca, instala los paquetes ejecutando la primera línea en una celda de Jupyter.


In [1]:
import re
import tarfile
import numpy as np
import pandas as pd
from pathlib import Path
from zipfile import ZipFile
from sklearn.feature_extraction.text import TfidfVectorizer

# El patrón conserva letras, números y apóstrofos internos.
TOKEN_RE = re.compile(r"[^\W_]+(?:['’][^\W_]+)?", flags=re.UNICODE)

def tokenize(text):
    # Paso a minúsculas y normalizo el apóstrofo curvo.
    text = text.lower().replace("’", "'")
    return TOKEN_RE.findall(text)


## Paso 2. Cargar 100 libros únicos

Aquí leo el ZIP que contiene un TAR interno. Recorro el TAR en modo secuencial, tomo un archivo TXT por cada ID de Gutenberg y me detengo al reunir 100 libros. El orden corresponde al archivo; no se carga toda la colección en memoria.


In [2]:
ZIP_PATH = Path(r"C:\Users\Estefano Condoy\Downloads\txt-files.tar.zip")

def load_first_unique_books(zip_path, limit=100):
    documents = {}

    with ZipFile(zip_path) as archive:
        # Encuentro el TAR que contiene los libros TXT.
        tar_name = next((name for name in archive.namelist()
                         if name.lower().endswith('.tar')), None)
        if tar_name is None:
            raise ValueError("El ZIP no contiene un archivo TAR")

        with archive.open(tar_name) as tar_stream:
            with tarfile.open(fileobj=tar_stream, mode='r|') as tar:
                for member in tar:
                    if not member.isfile() or not member.name.lower().endswith('.txt'):
                        continue

                    # Reconozco nombres como pg123.txt y pg123-0.txt.
                    match = re.match(
                        r"^(?:pg)?(\d+)(?:-\d+)?\.txt$",
                        Path(member.name).name,
                        flags=re.IGNORECASE
                    )
                    if not match:
                        continue

                    ebook_id = match.group(1)
                    # Si el ID ya apareció, omito esa versión duplicada.
                    if ebook_id in documents:
                        continue

                    stream = tar.extractfile(member)
                    if stream is None:
                        continue

                    documents[ebook_id] = stream.read().decode(
                        'utf-8', errors='replace'
                    )

                    if len(documents) == limit:
                        break

    return documents

documents = load_first_unique_books(ZIP_PATH, limit=100)
print("Libros cargados:", len(documents))
print("IDs Gutenberg de muestra:", list(documents)[:10])
if len(documents) < 100:
    print("Aviso: el archivo terminó antes de reunir 100 libros TXT únicos.")


Libros cargados: 100
IDs Gutenberg de muestra: ['10000', '10001', '10002', '10003', '10004', '10005', '10006', '10007', '10008', '10009']


## Paso 3. Calcular TF y DF

Aquí ajusto un `TfidfVectorizer` sin IDF ni normalización para obtener los conteos TF sin modificar. A partir de esa matriz, sumo los conteos para TF total y cuento los valores mayores que cero para DF.


In [3]:
texts = list(documents.values())

# Sin IDF y sin normalización, cada valor es el conteo del término en ese libro.
tf_vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
    use_idf=False,
    norm=None
)
tf_matrix = tf_vectorizer.fit_transform(texts)

tf_terms = tf_vectorizer.get_feature_names_out()
tf_total = np.asarray(tf_matrix.sum(axis=0)).ravel().astype(int)
df_total = np.asarray((tf_matrix > 0).sum(axis=0)).ravel().astype(int)

tf_df = pd.DataFrame({
    "término": tf_terms,
    "TF_total": tf_total,
    "DF": df_total
})

print("Forma de la matriz TF (documentos, términos):", tf_matrix.shape)
display(tf_df.sort_values("TF_total", ascending=False).head(20))


Forma de la matriz TF (documentos, términos): (100, 114137)


,término,TF_total,DF
100073,the,335195,100
5419,and,170741,100
70031,of,169323,100
101048,to,135660,100
1728,a,116545,100
51079,in,95294,100
50104,i,67705,97
100054,that,62628,100
47457,he,54016,100
53622,it,53525,100


## Paso 4. Calcular IDF y TF-IDF con la librería y manualmente

Aquí calculo el IDF de dos maneras para compararlos. Uso `smooth_idf=False` en `TfidfVectorizer`, de modo que la librería aplique la misma fórmula manual: $idf(t)=\log(N/df(t))+1$. Después se incorpora ese IDF al TF y se normalizan los vectores por documento. La [documentación oficial de scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) describe estos parámetros.


In [4]:
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
    use_idf=True,
    # Desactivo el suavizado para comparar con la fórmula manual solicitada.
    smooth_idf=False,
    norm="l2"
)
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)

tfidf_terms = tfidf_vectorizer.get_feature_names_out()
assert np.array_equal(tf_terms, tfidf_terms), "Los vocabularios de TF e IDF no coinciden"

# idf_ contiene el peso IDF de cada término en el mismo orden del vocabulario.
idf_values = tfidf_vectorizer.idf_
tfidf_total = np.asarray(tfidf_matrix.sum(axis=0)).ravel()

# Calculo manualmente IDF usando el DF obtenido en el Paso 3.
N = len(texts)

def calcular_idf(df, N):
    return np.log(N / df) + 1

idf_manual = calcular_idf(df_total, N)
print("IDF de la librería y manual coinciden:", np.allclose(idf_values, idf_manual))

# Muestro ambos resultados lado a lado para revisar la comparación.
idf_comparison = pd.DataFrame({
    "término": tf_terms,
    "DF": df_total,
    "IDF_librería": idf_values,
    "IDF_manual": idf_manual
})
display(idf_comparison.head(10).round(4))

term_stats = pd.DataFrame({
    "término": tf_terms,
    "TF_total": tf_total,
    "DF": df_total,
    "IDF": idf_values,
    "TF_IDF_total": tfidf_total
})

print("Forma de la matriz TF-IDF:", tfidf_matrix.shape)
print("Términos con TF más alto:")
display(term_stats.sort_values("TF_total", ascending=False).head(20))

print("Términos con IDF más alto (más raros en estos documentos):")
display(term_stats.sort_values("IDF", ascending=False).head(20))


IDF de la librería y manual coinciden: True


,término,DF,IDF_librería,IDF_manual
0,0,13,3.0402,3.0402
1,00,18,2.7148,2.7148
2,000,100,1.0000,1.0000
3,001,2,4.9120,4.9120
4,0014,1,5.6052,5.6052
5,0023,1,5.6052,5.6052
6,003,2,4.9120,4.9120
7,0035,1,5.6052,5.6052
8,004,1,5.6052,5.6052
9,0042,1,5.6052,5.6052


Forma de la matriz TF-IDF: (100, 114137)
Términos con TF más alto:


,término,TF_total,DF,IDF,TF_IDF_total
100073,the,335195,100,1.000000,56.494524
5419,and,170741,100,1.000000,28.420315
70031,of,169323,100,1.000000,30.039790
101048,to,135660,100,1.000000,23.737142
1728,a,116545,100,1.000000,19.767332
51079,in,95294,100,1.000000,16.167172
50104,i,67705,97,1.030459,10.927775
100054,that,62628,100,1.000000,10.591700
47457,he,54016,100,1.000000,8.534733
53622,it,53525,100,1.000000,8.938060


Términos con IDF más alto (más raros en estos documentos):


,término,TF_total,DF,IDF,TF_IDF_total
114136,ἀποκολοκύντωσις,2,1,5.60517,0.013611
114135,ἀποθέωσις,1,1,5.60517,0.006806
114134,μωροῦ,1,1,5.60517,0.006806
114133,θεοῦ,1,1,5.60517,0.006806
114132,ębre,1,1,5.60517,0.003307
114116,überzeugte,1,1,5.60517,0.000157
114115,überzeugt,14,1,5.60517,0.002191
114114,überzeuget,1,1,5.60517,0.000157
114113,überzeugen,10,1,5.60517,0.001565
114112,überwältigen,1,1,5.60517,0.000157


## Paso 5. Interpretación de los resultados

En esta ejecución se analizaron 100 libros y se obtuvieron 114.137 términos distintos. El término **the** tiene TF = 335.195 porque aparece muchas veces, DF = 100 porque está presente en todos los libros e IDF = 1, por lo que ayuda poco a distinguir un libro de otro. El término **i** tiene TF = 67.705, DF = 97 e IDF ≈ 1,0305: al faltar en tres libros, su IDF es un poco mayor.

En general, TF mide las repeticiones; DF mide en cuántos documentos aparece el término; e IDF aumenta cuando el DF disminuye. TF-IDF combina esas medidas y asigna más peso a términos frecuentes dentro de algunos documentos, pero poco comunes en el conjunto. Como no se eliminaron stopwords, palabras comunes como **the**, **and** y **of** están entre los términos con mayor TF. La comparación manual y la librería debe indicar `True` porque usan la misma fórmula y configuración.
